# 01 - Data Exploration

Explore and profile all financial NLP datasets before training.

**Run this notebook in Kaggle with GPU enabled.**

In [ ]:
%%capture
!pip install unsloth datasets rouge_score

In [ ]:
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB)")
else:
    print("No GPU (CPU-only mode - fine for data exploration)")

In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. FinGPT Sentiment Training Set

In [ ]:
ds_sentiment = load_dataset("FinGPT/fingpt-sentiment-train", split="train")
print(f"Rows: {len(ds_sentiment)}")
print(f"Columns: {ds_sentiment.column_names}")
df_sent = ds_sentiment.to_pandas()
df_sent.head()

In [ ]:
# Label distribution
print("\n=== Label Distribution ===")
print(df_sent['output'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_sent['output'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c', '#95a5a6'])
axes[0].set_title('FinGPT Sentiment - Label Distribution')
axes[0].set_ylabel('Count')

# Text length distribution
df_sent['text_len'] = df_sent['input'].str.len()
df_sent['text_len'].hist(bins=50, ax=axes[1], color='#3498db', alpha=0.7)
axes[1].set_title('FinGPT Sentiment - Text Length Distribution')
axes[1].set_xlabel('Character Count')
axes[1].axvline(df_sent['text_len'].median(), color='red', linestyle='--', label=f'Median: {df_sent["text_len"].median():.0f}')
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"\nText length stats:")
print(df_sent['text_len'].describe())

In [ ]:
# Sample examples per label
for label in df_sent['output'].unique():
    print(f"\n=== {label} ===")
    samples = df_sent[df_sent['output'] == label].sample(2, random_state=42)
    for _, row in samples.iterrows():
        print(f"  - {row['input'][:200]}...")

## 2. Financial PhraseBank

In [ ]:
ds_phrasebank = load_dataset("financial_phrasebank", "sentences_allagree", split="train")
print(f"Rows: {len(ds_phrasebank)}")
print(f"Columns: {ds_phrasebank.column_names}")
df_pb = ds_phrasebank.to_pandas()
df_pb.head()

In [ ]:
label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
df_pb['label_name'] = df_pb['label'].map(label_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_pb['label_name'].value_counts().plot(kind='bar', ax=axes[0], color=['#95a5a6', '#2ecc71', '#e74c3c'])
axes[0].set_title('Financial PhraseBank - Label Distribution')

df_pb['text_len'] = df_pb['sentence'].str.len()
df_pb['text_len'].hist(bins=40, ax=axes[1], color='#9b59b6', alpha=0.7)
axes[1].set_title('Financial PhraseBank - Text Length')
plt.tight_layout()
plt.show()

## 3. ConvFinQA (Numerical Reasoning)

In [ ]:
ds_convfinqa = load_dataset("FinGPT/ConvFinQA", split="train")
print(f"Rows: {len(ds_convfinqa)}")
print(f"Columns: {ds_convfinqa.column_names}")
df_cfq = ds_convfinqa.to_pandas()
df_cfq.head()

In [ ]:
# Question length distribution
df_cfq['q_len'] = df_cfq['question'].str.len()
df_cfq['q_len'].hist(bins=40, color='#e67e22', alpha=0.7)
plt.title('ConvFinQA - Question Length Distribution')
plt.xlabel('Character Count')
plt.show()

print(f"\nSample question: {df_cfq.iloc[0]['question'][:300]}")
print(f"Sample answer: {df_cfq.iloc[0]['answer']}")

## 4. Summary Statistics

In [ ]:
summary = pd.DataFrame({
    'Dataset': ['FinGPT Sentiment', 'Financial PhraseBank', 'ConvFinQA'],
    'Rows': [len(df_sent), len(df_pb), len(df_cfq)],
    'Task': ['Sentiment', 'Sentiment', 'Reasoning'],
    'Avg Text Length': [
        df_sent['text_len'].mean(),
        df_pb['text_len'].mean(),
        df_cfq['q_len'].mean(),
    ],
})
print(summary.to_string(index=False))
print(f"\nTotal raw examples: {summary['Rows'].sum()}")

## 5. Data Quality Check

In [ ]:
# Check for empty/null texts
print("Empty texts in FinGPT Sentiment:", df_sent['input'].isna().sum() + (df_sent['input'] == '').sum())
print("Empty texts in PhraseBank:", df_pb['sentence'].isna().sum() + (df_pb['sentence'] == '').sum())
print("Empty questions in ConvFinQA:", df_cfq['question'].isna().sum() + (df_cfq['question'] == '').sum())

# Check for very short texts (< 20 chars)
short_sent = (df_sent['input'].str.len() < 20).sum()
short_pb = (df_pb['sentence'].str.len() < 20).sum()
print(f"\nVery short (<20 chars) in Sentiment: {short_sent}")
print(f"Very short (<20 chars) in PhraseBank: {short_pb}")